# Predicting late deliveries
Goal: predict `late_delivery` at dispatch time. Follows `checklist.md`.

Decisions up front:
- `claim_paid_eur` is a compensation settled *after* delivery, so it is not known at prediction time (and is a near-proxy for the label). **Dropped.**
- `consignment` is an identifier. **Dropped.** Each consignment appears once, so entities are not repeated across rows.
- A model would be used on future dispatches, so the split is by time: the latest rows are the test set.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, brier_score_loss, precision_score, recall_score

df = pd.read_csv('data.csv', parse_dates=['dispatched_on']).sort_values('dispatched_on').reset_index(drop=True)
df['hour'] = df.dispatched_on.dt.hour
df['dow'] = df.dispatched_on.dt.dayofweek
print(df.shape, df.dispatched_on.min(), df.dispatched_on.max(), df.consignment.is_unique)
print('late rate', df.late_delivery.mean())

## 1. Set aside the test set (latest 20% of rows by dispatch time)

In [ ]:
cut = int(len(df)*0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(len(train), len(test), train.dispatched_on.max(), test.dispatched_on.min())
assert train.dispatched_on.max() <= test.dispatched_on.min()
FEATS = ['carrier','distance_km','weight_kg','parcels','hour','dow']   # all known at dispatch
y_tr, y_te = train.late_delivery.values, test.late_delivery.values
X_tr, X_te = train[FEATS], test[FEATS]
print('train late rate', y_tr.mean())

## 2-4. Candidates (named before any scoring)
1. `baseline`: predicts the training prior (trivial).
2. `logreg_C{0.1,1,10}`: logistic regression on scaled numerics + one-hot carrier.
3. `hgb_d{2,3}`: gradient boosting, `max_depth` 2 or 3, `learning_rate=0.05`, 150 iterations.

Selection metric: mean ROC AUC over expanding-window time-series CV folds on training rows only. The decision threshold is chosen on pooled out-of-fold predictions (maximising F1). All preprocessing sits inside the pipelines, so it is fitted per fold.

In [ ]:
num = ['distance_km','weight_kg','parcels','hour','dow']
def prep(scale):
    return ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', num),
                              ('cat', OneHotEncoder(handle_unknown='ignore'), ['carrier'])])
def make(name):
    if name=='baseline': return Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))])
    if name.startswith('logreg'):
        C_=float(name.split('_C')[1]); return Pipeline([('p',prep(True)),('m',LogisticRegression(C=C_,max_iter=1000))])
    d=int(name.split('_d')[1]); return Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=150,random_state=0))])
CANDIDATES = ['baseline','logreg_C0.1','logreg_C1','logreg_C10','hgb_d2','hgb_d3']

## Time-series cross-validation on training rows

In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
rows, oof = [], {}
for name in CANDIDATES:
    aucs, aps, p_all, y_all = [], [], [], []
    for tr, va in tscv.split(X_tr):
        m = make(name).fit(X_tr.iloc[tr], y_tr[tr])
        p = m.predict_proba(X_tr.iloc[va])[:,1]
        aucs.append(roc_auc_score(y_tr[va], p)); aps.append(average_precision_score(y_tr[va], p))
        p_all.append(p); y_all.append(y_tr[va])
    oof[name] = (np.concatenate(p_all), np.concatenate(y_all))
    rows.append((name, np.mean(aucs), np.std(aucs), np.mean(aps)))
cv = pd.DataFrame(rows, columns=['model','cv_auc','cv_auc_sd','cv_ap']).sort_values('cv_auc', ascending=False)
cv

## 5. Choose model and threshold on CV only

In [ ]:
best = cv.iloc[0].model
p, y = oof[best]
ths = np.linspace(0.05, 0.8, 76)
f1s = [f1_score(y, p>=t) for t in ths]
THRESH = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(THRESH,3), '| OOF F1:', round(max(f1s),3))
print('OOF precision/recall at threshold:', precision_score(y,p>=THRESH), recall_score(y,p>=THRESH))

### Written choice (fixed before the test set is touched)
The model with the highest mean CV AUC above is carried forward with the F1-optimal out-of-fold threshold printed above. If the best candidate does not clearly beat the baseline in CV (gap within about one fold SD), the honest conclusion is that late delivery is not predictable from these features. No further changes are made after seeing test results.

## 6. Refit on all training rows, score the test set once

In [ ]:
final = make(best).fit(X_tr, y_tr)
pt = final.predict_proba(X_te)[:,1]
base = make('baseline').fit(X_tr, y_tr).predict_proba(X_te)[:,1]
res = pd.DataFrame({
 'model':[best,'baseline'],
 'AUC':[roc_auc_score(y_te,pt), roc_auc_score(y_te,base)],
 'PR-AUC':[average_precision_score(y_te,pt), average_precision_score(y_te,base)],
 'Brier':[brier_score_loss(y_te,pt), brier_score_loss(y_te,base)]})
print(res.round(4).to_string(index=False))
pred = pt>=THRESH
print('threshold', round(THRESH,3), 'precision', round(precision_score(y_te,pred,zero_division=0),3),
      'recall', round(recall_score(y_te,pred),3), 'F1', round(f1_score(y_te,pred),3), '| test late rate', y_te.mean().round(3))

## Conclusion
Headline (single test-set score, latest 20% of dispatches, model `hgb_d2` chosen on CV): **AUC 0.68** vs 0.50 for the baseline. CV AUC was about 0.72, so the signal is real but modest and slightly weaker on the future rows.

Important caveat, observed after scoring and **not acted on**: the late rate fell from 25.7% in training to 2.2% in the test period, which is a large drift. The test PR-AUC is low (0.047, versus 0.022 for the baseline) and the CV-chosen threshold of 0.22 gives precision of only 0.035 on test, because it was tuned to a base rate that no longer holds. Rankings carry over somewhat, but the probabilities and the threshold do not. Before deployment, check why the late rate dropped (process change, promise-date policy, data issue) and recalibrate or retrain on recent data.

Other caveats: single temporal split; only dispatch-time features are used, so weather, congestion and promised lead time (not in the data) are unavailable.